<h3>In this notebook we'll be generating the synthetic dataset

In [1]:
import pandas as pd
import numpy as np

**Step 1: Construct the Base Grid & Calendar Flags**

In [2]:
dates = pd.date_range(start="2026-01-01 00:00:00", end="2026-12-31 23:45:00", freq="15min")

**Doing Cartesian Product**

In [3]:
station = ["Rajiv Chowk", "Noida Sector 62", "Hauz Khas", "Vishwavidyalaya", "Vaishali"]
df = pd.MultiIndex.from_product([station, dates], names=["Station", "Timestamp"]).to_frame(index=False)

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 175200 entries, 0 to 175199
Data columns (total 2 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   Station    175200 non-null  str           
 1   Timestamp  175200 non-null  datetime64[us]
dtypes: datetime64[us](1), str(1)
memory usage: 2.7 MB


**Extract helper time variables**

In [5]:
df["Fractionalhour"] = (df["Timestamp"].dt.hour + df["Timestamp"].dt.minute/60).astype(float)
df["Day"]= (df["Timestamp"].dt.dayofweek).astype(int) # (0 = "Mon", 6 = "Sun")
df["Month"] = (df["Timestamp"].dt.month).astype(int)

In [6]:
df.head(15)

,Station,Timestamp,Fractionalhour,Day,Month
0,Rajiv Chowk,2026-01-01 00:00:00,0.00,3,1
1,Rajiv Chowk,2026-01-01 00:15:00,0.25,3,1
2,Rajiv Chowk,2026-01-01 00:30:00,0.50,3,1
3,Rajiv Chowk,2026-01-01 00:45:00,0.75,3,1
4,Rajiv Chowk,2026-01-01 01:00:00,1.00,3,1
5,Rajiv Chowk,2026-01-01 01:15:00,1.25,3,1
6,Rajiv Chowk,2026-01-01 01:30:00,1.50,3,1
7,Rajiv Chowk,2026-01-01 01:45:00,1.75,3,1
8,Rajiv Chowk,2026-01-01 02:00:00,2.00,3,1
9,Rajiv Chowk,2026-01-01 02:15:00,2.25,3,1


In [7]:
df["is_weekend"] = (df["Timestamp"].dt.day_of_week >= 5).astype(int)

In [8]:
HOLIDAYS = {
    "2026-01-26",
    "2026-02-15",
    "2026-03-04",
    "2026-03-21",
    "2026-03-26",
    "2026-03-31",
    "2026-04-03",
    "2026-05-01",
    "2026-05-27",
    "2026-08-15",
    "2026-09-04",
    "2026-10-02",
    "2026-10-20",
    "2026-11-08",
    "2026-12-25",
}

df["IsHoliday"] = (df["Timestamp"].dt.strftime("%Y-%m-%d").isin(HOLIDAYS).astype(int))

**Build the Station Parameter Dictionary**

In [9]:
STATION_PARAMS = {
    "Rajiv Chowk": {
        "max_cap": 1500,
        "base_ratio": 0.35,
        "morn_peak_hr": 9.25,
        "morn_weight": 0.85,
        "eve_peak_hr": 18.50,
        "eve_weight": 1.00,
        "weekend_mult": 0.80,
    },
    "Noida Sector 62": {
        "max_cap": 900,
        "base_ratio": 0.15,
        "morn_peak_hr": 9.50,
        "morn_weight": 0.95,
        "eve_peak_hr": 18.75,
        "eve_weight": 0.90,
        "weekend_mult": 0.25,
    },
    "Hauz Khas": {
        "max_cap": 800,
        "base_ratio": 0.30,  # Commercial/interchange hub
        "morn_peak_hr": 9.00,
        "morn_weight": 0.65,
        "eve_peak_hr": 19.00,
        "eve_weight": 0.95,
        "weekend_mult": 1.15,
    },
    "Vishwavidyalaya": {
        "max_cap": 600,
        "base_ratio": 0.25,  # University daytime footfall
        "morn_peak_hr": 9.00,
        "morn_weight": 0.85,
        "eve_peak_hr": 15.50,
        "eve_weight": 0.80,
        "weekend_mult": 0.35,
    },
    "Vaishali": {
        "max_cap": 450,
        "base_ratio": 0.20,  # Residential commuter station
        "morn_peak_hr": 8.25,
        "morn_weight": 0.90,
        "eve_peak_hr": 19.50,
        "eve_weight": 0.85,
        "weekend_mult": 0.55,
    },
}


**Mapping Parameter keys with stations**

In [10]:
param_keys = [
    "max_cap",
    "base_ratio",
    "morn_peak_hr",
    "morn_weight",
    "eve_peak_hr",
    "eve_weight",
    "weekend_mult",
]

for key in param_keys:
    df[key] = df["Station"].map(lambda s: STATION_PARAMS[s][key])

**Compute the Expected Crowd Rate ($\lambda_{\text{base}}$)**

In [11]:
df.head()

,Station,Timestamp,Fractionalhour,Day,Month,is_weekend,IsHoliday,max_cap,base_ratio,morn_peak_hr,morn_weight,eve_peak_hr,eve_weight,weekend_mult
0,Rajiv Chowk,2026-01-01 00:00:00,0.00,3,1,0,0,1500,0.35,9.25,0.85,18.5,1.0,0.8
1,Rajiv Chowk,2026-01-01 00:15:00,0.25,3,1,0,0,1500,0.35,9.25,0.85,18.5,1.0,0.8
2,Rajiv Chowk,2026-01-01 00:30:00,0.50,3,1,0,0,1500,0.35,9.25,0.85,18.5,1.0,0.8
3,Rajiv Chowk,2026-01-01 00:45:00,0.75,3,1,0,0,1500,0.35,9.25,0.85,18.5,1.0,0.8
4,Rajiv Chowk,2026-01-01 01:00:00,1.00,3,1,0,0,1500,0.35,9.25,0.85,18.5,1.0,0.8


In [12]:
df["is_open"] = np.where(
    ((df["is_weekend"] == 0) & (df["Fractionalhour"] >= 5.5) & (df["Fractionalhour"] <= 23.5)) |
    ((df["is_weekend"] == 1) & (df["Fractionalhour"] >= 6.25) & (df["Fractionalhour"] <= 23.5)),
    1.0,
    0.005
)

**Creating Guassian Fuction** 

In [13]:
def gaussian(h, mu, sigma):
    return np.exp(-0.5 * ((h - mu) / sigma) ** 2)

In [14]:
# 1. Compute base morning and evening curves

h = df["Fractionalhour"]
morning_curve = df["morn_weight"] * gaussian(h, df["morn_peak_hr"], sigma=1.1)
evening_curve = df["eve_weight"] * gaussian(h, df["eve_peak_hr"], sigma=1.4)    

# 2. Dampen morning curve and apply weekend_mult on weekends or holidays

morning_curve = np.where((df["is_weekend"]) | (df["IsHoliday"] == 1), morning_curve * 0.25, morning_curve)
total_curve = morning_curve + evening_curve
total_curve = np.where((df["is_weekend"]) | (df["IsHoliday"] == 1), total_curve * df["weekend_mult"], total_curve)

# 3. Apply semester break multiplier (0.50) for Vishwavidyalaya in May, June, Dec

is_uni_break = (df["Station"] == "Vishwavidyalaya") & (df["Timestamp"].dt.month.isin([5, 6, 12]))
total_curve = np.where(is_uni_break, total_curve * 0.50, total_curve)

df["morning_curve"] = morning_curve
df["evening_curve"] = evening_curve
df["total_curve"] = total_curve

In [15]:
lambda_base = df["max_cap"] * df["is_open"] * (df["base_ratio"] + df["total_curve"])

In [16]:
# Set a seed for reproducibility (optional)
np.random.seed(42)

# 1. Initialize temporary shock multiplier array (all 1.0s)
shock_mult = np.ones(len(df), dtype=float)

# Extract normalized dates once for fast matching
all_dates = df["Timestamp"].dt.normalize().unique()

# 2. Loop over each station (only 5 iterations, very fast)
for stn in df["Station"].unique():
    # Pick 5 or 6 random dates out of 365 without replacement
    num_shock_days = np.random.choice([5, 6])
    chosen_dates = np.random.choice(all_dates, size=num_shock_days, replace=False)
    
    for d in chosen_dates:
        # Find row indices for this station on this date (96 slots of 15 mins)
        day_indices = df.index[
            (df["Station"] == stn) & (df["Timestamp"].dt.normalize() == d)
        ].to_numpy()
        
        # Pick a random starting slot so 12 consecutive slots stay within the day (0 to 84)
        start_slot = np.random.randint(0, len(day_indices) - 11)
        window_indices = day_indices[start_slot : start_slot + 12]
        
        # Randomly choose surge (1.4) or snag (0.5)
        multiplier = np.random.choice([1.4, 0.5])
        shock_mult[window_indices] = multiplier

# 3. Apply shocks directly to lambda_base (without saving shock_mult to df)
lambda_base = lambda_base * shock_mult


**Understanding the lambda_base**

In [17]:
print(lambda_base[32735])
print(lambda_base[32736])
print(lambda_base[33737])

2.6316286973020127
2.6250000000000027
1368.4312291532895


In [18]:
print(df.iloc[32735])
print(df.iloc[32736])
print(df.iloc[33737])

Station                   Rajiv Chowk
Timestamp         2026-12-07 23:45:00
Fractionalhour                  23.75
Day                                 0
Month                              12
is_weekend                          0
IsHoliday                           0
max_cap                          1500
base_ratio                       0.35
morn_peak_hr                     9.25
morn_weight                      0.85
eve_peak_hr                      18.5
eve_weight                        1.0
weekend_mult                      0.8
is_open                         0.005
morning_curve                     0.0
evening_curve                0.000884
total_curve                  0.000884
Name: 32735, dtype: object
Station                   Rajiv Chowk
Timestamp         2026-12-08 00:00:00
Fractionalhour                    0.0
Day                                 1
Month                              12
is_weekend                          0
IsHoliday                           0
max_cap                

**Add Momentum, Hidden Shocks & Count Noise**

In [19]:
# Ensure lambda_base is a NumPy array aligned with df
lambda_base_arr = np.asarray(lambda_base)
all_lambda_dynamic = []

# Process each station's 35,040-element slice one station at a time
for stn, grp in df.groupby("Station", sort=False):
    idx = grp.index.to_numpy()
    
    # Extract this station's 35,040 elements and station-specific max_cap
    stn_lambda_base = lambda_base_arr[idx]
    stn_is_open = grp["is_open"].to_numpy()
    max_cap = STATION_PARAMS[stn]["max_cap"]  # or grp["max_cap"].iloc[0]
    
    # Initialize error and empty output list for this station
    error = 0.0
    lambda_dynamic = []
    
    # Loop through index t from 0 to 35,039
    for t in range(len(stn_lambda_base)):
        if stn_is_open[t] == 1.0:
            eps = np.random.normal(0, 0.07 * max_cap)
            error = 0.78 * error + eps
        else:
            # Keep error = 0 overnight so night noise doesn't bleed into 5:30 AM
            error = 0.0
            
        lambda_dynamic.append(max(0.0, stn_lambda_base[t] + error))
        
    all_lambda_dynamic.extend(lambda_dynamic)

# Convert combined list to a NumPy array (175,200 elements)
lambda_dynamic = np.array(all_lambda_dynamic)


In [20]:
df["observed_people_count"] = np.random.poisson(lam=lambda_dynamic)

In [21]:
df.tail(5)

,Station,Timestamp,Fractionalhour,Day,Month,is_weekend,IsHoliday,max_cap,base_ratio,morn_peak_hr,morn_weight,eve_peak_hr,eve_weight,weekend_mult,is_open,morning_curve,evening_curve,total_curve,observed_people_count
175195,Vaishali,2026-12-31 22:45:00,22.75,3,12,0,0,450,0.2,8.25,0.9,19.5,0.85,0.55,1.000,1.669806e-38,0.057439,0.057439,71
175196,Vaishali,2026-12-31 23:00:00,23.00,3,12,0,0,450,0.2,8.25,0.9,19.5,0.85,0.55,1.000,8.135065e-40,0.037346,0.037346,66
175197,Vaishali,2026-12-31 23:15:00,23.25,3,12,0,0,450,0.2,8.25,0.9,19.5,0.85,0.55,1.000,3.763772e-41,0.023520,0.023520,92
175198,Vaishali,2026-12-31 23:30:00,23.50,3,12,0,0,450,0.2,8.25,0.9,19.5,0.85,0.55,1.000,1.653686e-42,0.014348,0.014348,38
175199,Vaishali,2026-12-31 23:45:00,23.75,3,12,0,0,450,0.2,8.25,0.9,19.5,0.85,0.55,0.005,6.900018e-44,0.008478,0.008478,0


In [22]:
occ_ratio = df["observed_people_count"]/df["max_cap"]
df["CrowdStatus"] = np.select([occ_ratio < 0.30, occ_ratio < 0.65], ["Low", "Medium"], default="High")

In [23]:
df.to_csv("Fulldata.csv", index=False)

In [24]:
colstokeep = ["Timestamp", "Station", "is_weekend", "IsHoliday", "observed_people_count", "CrowdStatus"]
finaldf = df[colstokeep]

In [25]:
finaldf.to_csv("delhi_metro_crowd_2026.csv", index=False)